In [1]:
#Apply Raw
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time

# Load dataset
df= pd.read_csv("data/AmesHousing_engineered.csv")

# Drop target and ID columns
X_raw = df.drop(columns=["SalePrice", "PID", "Order"], errors="ignore")
print("Features shape (raw version):", X_raw.shape)

#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch



Features shape (raw version): (2930, 172)


In [2]:
#K-Means
start_time = time.time()
kmeans_raw = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    kmeans.fit(X_raw)
    labels = kmeans.labels_
    sil, db, ch = compute_metrics(X_raw, labels) #K-Means on Raw Featureslabels)
    kmeans_raw.append({"algorithm":"K-Means","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")
pd.DataFrame(kmeans_raw).round(4)

Runtime: 12.104504346847534 seconds
K-Means runtime: 12.1045 seconds


,algorithm,preprocessing,k,silhouette,davies_bouldin,calinski_harabasz
0,K-Means,raw,2,0.3934,1.0484,2555.8623
1,K-Means,raw,3,0.3220,1.1663,2059.4929
2,K-Means,raw,4,0.2870,1.2606,1835.8979
3,K-Means,raw,5,0.2715,1.4524,1627.8574
4,K-Means,raw,6,0.2642,1.4749,1438.7504
5,K-Means,raw,7,0.2555,1.5639,1313.8565
6,K-Means,raw,8,0.2138,1.6936,1195.0706


In [3]:
#GMM on Raw Features
start_time = time.time()
gmm_raw = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    gmm.fit(X_raw)
    labels = gmm.predict(X_raw)
    sil, db, ch = compute_metrics(X_raw, labels)
    gmm_raw.append({"algorithm":"GMM","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")

Runtime: 241.4003484249115 seconds
GMM runtime: 241.4003 seconds


In [4]:
#Agglomerative Clustering
start_time = time.time()
agg_raw = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    agg.fit(X_raw)
    labels = agg.labels_
    sil, db, ch = compute_metrics(X_raw, labels)
    agg_raw.append({"algorithm":"Agglomerative","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Runtime: 7.542431116104126 seconds
Agglomerative runtime: 7.5424 seconds


In [5]:
#Spectral Clustering
start_time = time.time()
spectral_raw = []
for k in k_values:
    spectral = SpectralClustering(n_clusters=k, affinity='nearest_neighbors', n_init=n_init, random_state=42)
    spectral.fit(X_raw)
    labels = spectral.labels_
    sil, db, ch = compute_metrics(X_raw, labels)
    spectral_raw.append({"algorithm":"Spectral","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")

Runtime: 4.884382724761963 seconds
Spectral runtime: 4.8844 seconds


In [6]:
#DBSCAN
start_time = time.time()
dbscan_raw = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    dbscan.fit(X_raw)
    labels = dbscan.labels_
    sil, db, ch = compute_metrics(X_raw, labels)
    dbscan_raw.append({"algorithm":"DBSCAN","preprocessing":"raw","eps":eps,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds") 

Runtime: 0.8617963790893555 seconds
DBSCAN runtime: 0.8618 seconds


In [7]:
start_time = time.time()
from sklearn.cluster import Birch

birch_raw = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_raw)

    if len(set(labels)) > 1:
        sil, db, ch = compute_metrics(X_raw, labels)
        birch_raw.append({
            "algorithm": "BIRCH",
            "preprocessing": "raw",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Birch runtime: {runtime:.4f} seconds")

Runtime: 2.97062611579895 seconds
Birch runtime: 2.9706 seconds


In [8]:
start_time = time.time()
from sklearn.cluster import OPTICS

optics_raw = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_raw)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_raw, labels)
        optics_raw.append({
            "algorithm": "OPTICS",
            "preprocessing": "raw",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


Runtime: 55.63440561294556 seconds
Optics runtime: 55.6344 seconds


In [9]:
import csv

ames_results_raw = (kmeans_raw + gmm_raw + agg_raw + spectral_raw + dbscan_raw + birch_raw + optics_raw)

# Desired column order
keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]
with open('updated_data/ames_data/ames_raw.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(ames_results_raw)

In [11]:
from sklearn.metrics import adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

#  ARI stability analysis 
n_bootstrap = 100
ari_results = []
# Collect all parameter settings from your previous results
all_configs = []

for r in kmeans_raw:
    all_configs.append(("K-Means", {"k": r["k"]}))

for r in gmm_raw:
    all_configs.append(("GMM", {"k": r["k"]}))

for r in agg_raw:
    all_configs.append(("Agglomerative", {"k": r["k"]}))

for r in spectral_raw:
    all_configs.append(("Spectral", {"k": r["k"]}))

for r in dbscan_raw:
    all_configs.append(("DBSCAN", {"eps": r["eps"]}))

for r in birch_raw:
    all_configs.append(("BIRCH", {"threshold": r["threshold"]}))

for r in optics_raw:
    all_configs.append(("OPTICS", {"min_samples": r["min_samples"]}))
# --- helper function to fit a model and return labels ---
def fit_and_predict(name, params, X_data):

    if name == "K-Means":
        model = KMeans(n_clusters=params["k"], n_init=n_init, random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "GMM":
        model = GaussianMixture(n_components=params["k"], n_init=n_init, random_state=42)
        labels = model.fit(X_data).predict(X_data)

    elif name == "Agglomerative":
        model = AgglomerativeClustering(n_clusters=params["k"], linkage='ward')
        labels = model.fit_predict(X_data)

    elif name == "Spectral":
        model = SpectralClustering(
            n_clusters=params["k"],
            affinity='nearest_neighbors',
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "DBSCAN":
        model = DBSCAN(eps=params["eps"], min_samples=min_samples)
        labels = model.fit_predict(X_data)

    elif name == "BIRCH":
        model = Birch(n_clusters=None, threshold=params["threshold"])
        labels = model.fit_predict(X_data)

    elif name == "OPTICS":
        model = OPTICS(min_samples=params["min_samples"], xi=0.05, n_jobs=-1)
        labels = model.fit_predict(X_data)

    else:
        return None

    return labels


# Reuse all parameter configurations from previous section
for algo_name, params in all_configs:

    # reference clustering on full data
    ref_labels = fit_and_predict(algo_name, params, X_raw)

    if ref_labels is None:
        continue

    ari_scores = []
    rng = np.random.RandomState(42)

    for b in range(n_bootstrap):

        # bootstrap sample with indices
        indices = rng.choice(len(X_raw), size=len(X_raw), replace=True)
        X_boot = X_raw.iloc[indices]

        boot_labels = fit_and_predict(algo_name, params, X_boot)

        if boot_labels is None:
            continue

        # compare only sampled observations
        ref_subset = np.array(ref_labels)[indices]

        # remove noise points for DBSCAN / OPTICS
        mask = (boot_labels != -1) & (ref_subset != -1)

        if np.sum(mask) < 2:
            continue

        ari = adjusted_rand_score(ref_subset[mask], np.array(boot_labels)[mask])
        ari_scores.append(ari)

    if len(ari_scores) > 0:
        ari_results.append({
            "algorithm": algo_name,
            **params,
            "ARI_mean": np.mean(ari_scores),
            "ARI_std": np.std(ari_scores)
        })


# Summary table
ari_df = pd.DataFrame(ari_results).round(4)

print("\nBOOTSTRAP ARI STABILITY")
print(ari_df.to_string(index=False))

# Top 3 most stable by ARI
top3_ari = ari_df.nlargest(3, "ARI_mean")

print("\n TOP 3 MOST STABLE BY ARI ")
print(top3_ari.to_string(index=False))

c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\S


BOOTSTRAP ARI STABILITY
    algorithm   k  ARI_mean  ARI_std  eps  threshold  min_samples
      K-Means 2.0    1.0000   0.0000  NaN        NaN          NaN
      K-Means 3.0    0.9837   0.0100  NaN        NaN          NaN
      K-Means 4.0    0.9927   0.0065  NaN        NaN          NaN
      K-Means 5.0    0.9945   0.0034  NaN        NaN          NaN
      K-Means 6.0    0.9205   0.0584  NaN        NaN          NaN
      K-Means 7.0    0.9291   0.0429  NaN        NaN          NaN
      K-Means 8.0    0.7914   0.0814  NaN        NaN          NaN
          GMM 2.0    0.6822   0.3129  NaN        NaN          NaN
          GMM 3.0    0.7223   0.2380  NaN        NaN          NaN
          GMM 4.0    0.4774   0.1806  NaN        NaN          NaN
          GMM 5.0    0.4670   0.0818  NaN        NaN          NaN
          GMM 6.0    0.6826   0.1506  NaN        NaN          NaN
          GMM 7.0    0.5041   0.0625  NaN        NaN          NaN
          GMM 8.0    0.6270   0.1327  NaN        Na

In [12]:
#the top three by stability score
ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

top_3 = (
    ari_df
    .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
    .head(3)
    .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
)

print(top_3.round(4).to_string(index=False))

algorithm   k  ARI_mean  ARI_std  Stability Score
  K-Means 2.0    1.0000   0.0000           1.0000
   DBSCAN NaN    0.9979   0.0014           0.9972
  K-Means 5.0    0.9945   0.0034           0.9932


In [13]:
#the best three clustering results overall, sort primarily by ARI_mean
top_3 = ari_df.sort_values(
    ["ARI_mean", "Stability Score"],
    ascending=[False, False]
).head(3)

print(top_3.round(4).to_string(index=False))

algorithm   k  ARI_mean  ARI_std  eps  threshold  min_samples  Stability Score
  K-Means 2.0    1.0000   0.0000  NaN        NaN          NaN           1.0000
   DBSCAN NaN    0.9979   0.0014  1.0        NaN          NaN           0.9972
   OPTICS NaN    0.9951   0.0143  NaN        NaN         20.0           0.9714


In [15]:
ari_df.to_csv("updated_data/ARI_Score/ames_raw_ari.csv", index=False)

In [9]:
#  Combine all algorithm results 
all_results = (
    kmeans_raw +
    gmm_raw +
    agg_raw +
    spectral_raw +
    dbscan_raw +
    birch_raw +
    optics_raw
)

results_df = pd.DataFrame(all_results)

# Round metric values to 4 decimal places
metric_cols = ["silhouette", "davies_bouldin", "calinski_harabasz"]
results_df[metric_cols] = results_df[metric_cols].round(4)

# Columns that may exist depending on algorithm
possible_cols = ["algorithm", "k", "eps", "threshold", "min_samples", "n_clusters"]

def available_cols(df, metric):
    cols = [c for c in possible_cols if c in df.columns]
    cols.append(metric)
    return cols

# Top 3 by Silhouette (higher is better) 
top3_sil = results_df.nlargest(3, "silhouette")

print("\n TOP 3 SILHOUETTE ")
print(top3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

# Top 3 by Davies-Bouldin (lower is better) 
top3_db = results_df.nsmallest(3, "davies_bouldin")

print("\n TOP 3 DAVIES-BOULDIN ")
print(top3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Top 3 by Calinski-Harabasz (higher is better)
top3_ch = results_df.nlargest(3, "calinski_harabasz")

print("\nTOP 3 CALINSKI-HARABASZ")
print(top3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))

# Bottom 3 by Silhouette (lower is worse) 
bottom3_sil = results_df.nsmallest(3, "silhouette")

print("\n BOTTOM 3 SILHOUETTE")
print(bottom3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Bottom 3 by Davies-Bouldin (higher is worse)
bottom3_db = results_df.nlargest(3, "davies_bouldin")

print("\nBOTTOM 3 DAVIES-BOULDIN ")
print(bottom3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Bottom 3 by Calinski-Harabasz (lower is worse)
bottom3_ch = results_df.nsmallest(3, "calinski_harabasz")

print("\nBOTTOM 3 CALINSKI-HARABASZ")
print(bottom3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))


 TOP 3 SILHOUETTE 
    algorithm   k  eps  threshold  min_samples  n_clusters  silhouette
      K-Means 2.0  NaN        NaN          NaN         NaN      0.3934
     Spectral 2.0  NaN        NaN          NaN         NaN      0.3927
Agglomerative 2.0  NaN        NaN          NaN         NaN      0.3772

 TOP 3 DAVIES-BOULDIN 
algorithm   k  eps  threshold  min_samples  n_clusters  davies_bouldin
    BIRCH NaN  NaN        0.2          NaN      2691.0          0.2029
    BIRCH NaN  NaN        0.5          NaN      1521.0          0.6102
 Spectral 3.0  NaN        NaN          NaN         NaN          1.0413

TOP 3 CALINSKI-HARABASZ
    algorithm   k  eps  threshold  min_samples  n_clusters  calinski_harabasz
      K-Means 2.0  NaN        NaN          NaN         NaN          2555.8623
     Spectral 2.0  NaN        NaN          NaN         NaN          2547.3105
Agglomerative 2.0  NaN        NaN          NaN         NaN          2422.2756

 BOTTOM 3 SILHOUETTE
algorithm   k  eps  threshold

In [10]:

# TOP 3 RESULTS FOR EACH ALGORITHM INDIVIDUALLY



all_algorithms = {
    "K-Means": kmeans_raw,
    "GMM": gmm_raw,
    "Agglomerative": agg_raw,
    "Spectral": spectral_raw,
    "DBSCAN": dbscan_raw,
    "BIRCH": birch_raw,
    "OPTICS": optics_raw
}



for algorithm, results in all_algorithms.items():

    if len(results) == 0:
        continue

    result_df = pd.DataFrame(results)


    # Round all validation values to 4 decimal places

    result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ] = result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ].round(4)



    print("\n")
    
    print(algorithm)
   



    # Select parameter column
  

    parameter_columns = [
        "k",
        "eps",
        "threshold",
        "min_samples"
    ]


    parameter = None

    for col in parameter_columns:
        if col in result_df.columns:
            parameter = col
            break




    # TOP 3 SILHOUETTE


    print("\nTop 3 Silhouette Score (Higher is better)")

    top_sil = result_df.nlargest(
        3,
        "silhouette"
    )

    print(
        top_sil[
            [
                parameter,
                "silhouette"
            ]
        ].to_string(index=False)
    )



   
    # TOP 3 DAVIES-BOULDIN
   

    print("\nTop 3 Davies-Bouldin Index (Lower is better)")

    top_db = result_df.nsmallest(
        3,
        "davies_bouldin"
    )

    print(
        top_db[
            [
                parameter,
                "davies_bouldin"
            ]
        ].to_string(index=False)
    )



    
    # TOP 3 CALINSKI-HARABASZ
    print("\nTop 3 Calinski-Harabasz Index (Higher is better)")

    top_ch = result_df.nlargest(
        3,
        "calinski_harabasz"
    )

    print(
        top_ch[
            [
                parameter,
                "calinski_harabasz"
            ]
        ].to_string(index=False)
    )



K-Means

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.3934
 3      0.3220
 4      0.2870

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.0484
 3          1.1663
 4          1.2606

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2          2555.8623
 3          2059.4929
 4          1835.8979


GMM

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 6      0.1541
 4      0.1447
 3      0.1413

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 5          2.2855
 4          2.3784
 3          2.6298

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 3           938.1876
 6           844.2316
 4           783.7065


Agglomerative

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.3772
 3      0.3146
 4      0.2460

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.0791
 3          1.1249
 4          1.2742

Top 3 Calinski-H